# Digital Twin GraphQL Exploration
Este cuaderno permite probar el backend de FastAPI/Strawberry a través del endpoint GraphQL.


In [2]:
pip install requests 

  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached urllib3-2.5.0-py3-none-any.whl.metadata (6.5 kB)
Using cached requests-2.32.5-py3-none-any.whl (64 kB)
Using cached urllib3-2.5.0-py3-none-any.whl (129 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [requests]

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import json
import requests

API_BASE_URL = os.getenv("API_BASE_URL", "http://localhost:8000/graphql")
print(f'Usando endpoint GraphQL: {API_BASE_URL}')


Usando endpoint GraphQL: http://localhost:8000/graphql


## Helper para ejecutar queries GraphQL
La función `run_query` envía queries al endpoint configurado y maneja errores comunes.


In [4]:
def run_query(query: str, variables: dict | None = None):
    response = requests.post(
        API_BASE_URL,
        json={'query': query, 'variables': variables},
        timeout=30,
    )
    response.raise_for_status()
    payload = response.json()
    if 'errors' in payload:
        raise RuntimeError(payload['errors'])
    return payload['data']


## Snapshot solar actual
Consulta los datos en tiempo real y métricas resumidas del gemelo digital.


In [5]:
solar_query = '''
query GetSolarSnapshot {
  solar {
    timestamp
    mode
    current {
      timestamp
      production
      consumption
      batteryLevel
      gridExport
      gridImport
    }
    metrics {
      currentProduction
      currentConsumption
      energyBalance
      systemEfficiency
    }
    weather {
      temperature
      solarRadiation
      cloudCover
      provider
    }
  }
}
'''
solar_data = run_query(solar_query)
solar_data


{'solar': {'timestamp': '2025-11-11T21:37:56.545994',
  'mode': 'predictive',
  'current': {'timestamp': '2025-11-11T21:37:56.545721',
   'production': 0.0,
   'consumption': 45.5,
   'batteryLevel': 9.5,
   'gridExport': 0.0,
   'gridImport': 0.0},
  'metrics': {'currentProduction': 0.0,
   'currentConsumption': 45.5,
   'energyBalance': -45.5,
   'systemEfficiency': 87.5},
  'weather': {'temperature': 14.6,
   'solarRadiation': 0.0,
   'cloudCover': 21.0,
   'provider': 'Simulación interna (fallback)'}}}

## Predicciones y alertas
Devuelve el bundle completo generado por `get_predictions_bundle`.


In [ ]:
predictions_query = '''
query GetPredictions {
  predictions {
    timestamp
    recommendations
    alerts {
      id
      type
      title
      message
    }
    predictions {
      hour
      expectedProduction
      expectedConsumption
      confidence
    }
    blackouts {
      date
      intervals {
        start
        end
      }
    }
  }
}
'''
predictions_data = run_query(predictions_query)
predictions_data


## Gestión de paneles y baterías
Ejemplo de cómo consultar la lista y detalles individuales.


In [6]:
list_panels_query = "
query ListPanels {
  panels {
    id_
    name
    ratedPowerKw
    quantity
  }
}
"
panels = run_query(list_panels_query)['panels']
panels


SyntaxError: unterminated string literal (detected at line 1) (3588426264.py, line 1)

In [ ]:
if panels:
    panel_query = "
    query PanelById($id: String!) {
      panel(id: $id) {
        name
        manufacturer
        model
        ratedPowerKw
        notes
      }
    }
    "
    first_id = panels[0]['id_']
    panel_detail = run_query(panel_query, {'id': first_id})
    panel_detail
else:
    print('No hay paneles en la base de datos todavía.')
